In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA = Path.cwd().parent / "data"
DAYS = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]
BUCKETS = [("not used", 0, "#eb6834"), ("used once", 1, "#2a78d6"), ("used twice", 2, "#1baf7a")]
SURFACE, INK, INK_2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK_2, "axes.titlecolor": INK,
    "axes.titlesize": 12, "axes.titlelocation": "left", "axes.titlepad": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelcolor": INK_2, "ytick.labelcolor": INK_2,
    "font.size": 10,
})

In [ ]:
vehicles = pd.read_csv(DATA / "General Data/vehicles.csv").rename(columns=str.strip)
orders = pd.concat([pd.read_csv(DATA / "Training Data/deliveries_train.csv"),
                    pd.read_csv(DATA / "Test Data/task1_test_inputs.csv")], ignore_index=True)
ran = orders[orders.dispatch_status == "attempted"]
days = pd.Index(sorted(ran.dispatch_date.unique()), name="date")
reefers = vehicles[vehicles.temp == "reefer"]
plio = vehicles[vehicles.depot == "Peliyagoda"]
plio_reefers = reefers[reefers.depot == "Peliyagoda"]
print(f"{len(ran):,} attempted orders, {ran.route_id.nunique():,} routes, {len(days)} operating days")

In [ ]:
scopes = [
    (vehicles, None, False, "Vehicle use by day of week: whole company (all vehicles)"),
    (plio, "Peliyagoda", False, "Vehicle use by day of week: Peliyagoda depot (all vehicles)"),
    (plio, "Peliyagoda", True, "Vehicle use by day of week: Peliyagoda chilled routes (all vehicles)"),
    (reefers, None, False, "Reefer use by day of week: whole company"),
    (plio_reefers, "Peliyagoda", False, "Reefer use by day of week: Peliyagoda depot"),
    (plio_reefers, "Peliyagoda", True, "Reefer use by day of week: Peliyagoda chilled routes"),
]
for fleet, depot, chilled_only, title in scopes:
    o = ran
    if depot:
        o = o[o.depot == depot]
    if chilled_only:
        o = o[o.temp_requirement == "chilled"]
    counts = (o[o.vehicle_id.isin(fleet.vehicle_id)].groupby(["dispatch_date", "vehicle_id"]).route_id.nunique()
              .unstack(fill_value=0).reindex(index=days, columns=fleet.vehicle_id, fill_value=0))
    per_day = pd.DataFrame({name: (counts == n).sum(axis=1) if n < 2 else (counts >= 2).sum(axis=1)
                            for name, n, _ in BUCKETS})
    per_day["dow"] = pd.to_datetime(per_day.index).dayofweek
    avg = per_day.groupby("dow")[[b for b, _, _ in BUCKETS]].mean().reindex(range(6))

    n = len(fleet)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), sharey=True)
    x = np.arange(6)
    for ax, (bucket, _, color) in zip(axes, BUCKETS):
        bars = ax.bar(x, avg[bucket].values, color=color, width=0.62)
        ax.bar_label(bars, labels=[f"{v:.1f}" for v in avg[bucket].values], padding=3, fontsize=9, color=INK_2)
        ax.axhline(n, color=INK_2, lw=1, ls="--")
        ax.set_xticks(x, DAYS)
        ax.grid(axis="x", visible=False)
        ax.set_title(f"Vehicles {bucket} (average {per_day[bucket].mean():.1f} a day)")
    axes[0].set_ylabel("Vehicles per day (average)")
    axes[0].set_ylim(0, n * 1.12)
    axes[2].annotate(f"fleet: {n} vehicles", (5.4, n), xytext=(0, 4), textcoords="offset points",
                     ha="right", va="bottom", fontsize=8.5, color=INK_2)
    fig.suptitle(title, x=0.07, ha="left", fontsize=14, color=INK, y=1.03)
    plt.show()
    print(avg.round(2).set_axis(DAYS).to_string())

In [ ]:
used_days = ran[ran.vehicle_id.isin(plio_reefers.vehicle_id)].groupby("vehicle_id").dispatch_date.nunique()
plio_reefers.set_index("vehicle_id")[["type", "weight_cap_kg", "volume_cap_m3"]].assign(
    days_used=used_days, share=(used_days / len(days)).round(2))

In [ ]:
sorted(set(vehicles.vehicle_id) - set(ran.vehicle_id))